# CD-A no-epoch with Weak-Faithfulness Screening

Verify and benchmark `dag_coordinate_descent_l0_weakfaith` (see
`docs/weak_faithfulness_cd_A_noepoch_zh.md`). The baseline is
`coordinate0.dag_coordinate_descent_l0` (cd_A_noepoch).

Sections:

1. Environment and imports
2. Regression test: `faithfulness_tau=0` must match the baseline byte-for-byte
3. Mask sparsity: how many pairs survive under `corr` vs `pcorr` for typical ER graphs
4. Path-cancellation sanity check (d=4)
5. Performance scan skeleton over (screening, tau, sampling_mode)

In [ ]:
# 1) Environment and imports
import os
import sys
import time

import numpy as np
import pandas as pd

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

from coordinate_descent.coordinate0 import dag_coordinate_descent_l0
from coordinate_descent.cd_A_weakfaith import (
    dag_coordinate_descent_l0_weakfaith,
    _build_faithfulness_mask,
)
from synthetic_dataset import SyntheticDataset

print('cd_A          :', 'OK' if callable(dag_coordinate_descent_l0) else 'MISSING')
print('cd_A_weakfaith:', 'OK' if callable(dag_coordinate_descent_l0_weakfaith) else 'MISSING')

## 2. Regression: `faithfulness_tau=0` ≡ baseline

With `faithfulness_tau=0` the new function must take the `allowed_offdiag is None`
branch and call `np.random.choice(d, 2, replace=True)` with the same RNG state
as the baseline. Outputs must be identical element-wise.

In [ ]:
# 2) tau=0 regression on several (d, seed) combinations
regression_cases = []
for d_val, seed_val in [(10, 0), (20, 1), (30, 7)]:
    ds = SyntheticDataset(n=2000, d=d_val, graph_type='ER', degree=2.0,
                          noise_type='gaussian_ev', B_scale=1.0, seed=seed_val)
    S = ds.X.T @ ds.X / ds.X.shape[0]

    r_old = dag_coordinate_descent_l0(
        S, T=1000, seed=42, return_history=True,
    )
    r_new = dag_coordinate_descent_l0_weakfaith(
        S, T=1000, seed=42, return_history=True,
        faithfulness_tau=0.0,
    )

    ok = (
        np.array_equal(r_old[0], r_new[0])
        and np.array_equal(r_old[1], r_new[1])
        and r_old[2] == r_new[2]
        and r_old[3] == r_new[3]
    )
    regression_cases.append({'d': d_val, 'seed': seed_val, 'match': ok})
    assert ok, f'tau=0 regression FAILED at d={d_val}, seed={seed_val}'

pd.DataFrame(regression_cases)

## 3. Mask sparsity: `corr` vs `pcorr`

Count the number of allowed off-diagonal pairs $M$ relative to the total
$d(d-1)$ for several threshold values on an ER graph. Expect `pcorr` to
prune more aggressively (smaller $M$), since the precision matrix's
non-zero pattern corresponds to the moral graph rather than the transitive
closure of directed paths.

In [ ]:
# 3) Mask sparsity on a fixed ER graph
ds = SyntheticDataset(n=20000, d=30, graph_type='ER', degree=2.0,
                      noise_type='gaussian_ev', B_scale=1.0, seed=0)
S = ds.X.T @ ds.X / ds.X.shape[0]
d = S.shape[0]
total = d * (d - 1)

records = []
for screening in ('corr', 'pcorr'):
    for tau in (0.0, 0.02, 0.05, 0.1, 0.15):
        _, M = _build_faithfulness_mask(S, tau, screening) if tau > 0 else (None, total)
        records.append({'screening': screening, 'tau': tau,
                        'M': M, 'M/total': M / total})

pd.DataFrame(records).pivot(index='tau', columns='screening', values='M/total')

## 4. Path-cancellation sanity check (d=4)

Construct the canonical failure case: $A\to B\to C\to D$ plus a direct
$A\to D$ edge whose weight is chosen so the two paths cancel in population
($\operatorname{corr}(A, D) \approx 0$).

- `tau = 0` should recover the $A\to D$ edge.
- `tau > 0, screening='corr'` should drop it (expected theoretical cost).
- `screening='pcorr'` may also drop it if $\Omega_{AD}$ is small; recorded for observation.

In [ ]:
# 4) Path-cancellation test
# X = B^T X + eps, i.e. X_j = sum_i B[i,j] X_i + eps_j
# Chain: 0 -> 1 -> 2 -> 3 with coefficient alpha each.
# Direct edge 0 -> 3 with coefficient -alpha^3 cancels the chain in expectation.
alpha = 0.6
direct = -alpha ** 3
B_true = np.zeros((4, 4))
B_true[0, 1] = alpha
B_true[1, 2] = alpha
B_true[2, 3] = alpha
B_true[0, 3] = direct

n = 200000
rng = np.random.RandomState(0)
eps = rng.randn(n, 4)
X = np.zeros((n, 4))
for j in range(4):
    X[:, j] = eps[:, j]
    for i in range(j):
        X[:, j] += B_true[i, j] * X[:, i]
S_path = X.T @ X / n
corr_mat = S_path / np.sqrt(np.outer(np.diag(S_path), np.diag(S_path)))
print('Empirical corr(0, 3) =', corr_mat[0, 3])

def recovered_edge(G):
    return bool(G[0, 3]) or bool(G[3, 0])

rows = []
for label, kwargs in [
    ('tau=0 (baseline)', {'faithfulness_tau': 0.0}),
    ('tau=0.05, corr',   {'faithfulness_tau': 0.05, 'screening': 'corr'}),
    ('tau=0.05, pcorr',  {'faithfulness_tau': 0.05, 'screening': 'pcorr'}),
]:
    _, G, obj = dag_coordinate_descent_l0_weakfaith(
        S_path, T=5000, seed=3, lambda_l0=0.1, **kwargs,
    )
    rows.append({'config': label, 'recovered_0->3_or_3->0': recovered_edge(G),
                 'final_f': round(obj, 4)})

pd.DataFrame(rows)

## 5. Performance scan (skeleton)

A single-configuration walkthrough to sanity-check timing and SHD.
Expand the grid (more `d`, more trials) before using for real benchmarks.

In [ ]:
# 5) Small-scale performance smoke: d=30, one trial per config
def shd_score(G_true, G_est):
    # Lower-bound SHD: number of differing directed edges (ignores CPDAG).
    return int(np.sum(np.abs(G_true.astype(int) - G_est.astype(int))))

ds = SyntheticDataset(n=20000, d=30, graph_type='ER', degree=2.0,
                      noise_type='gaussian_ev', B_scale=1.0, seed=0)
S = ds.X.T @ ds.X / ds.X.shape[0]
B_true = (np.abs(ds.B) > 0).astype(int)

configs = [
    {'faithfulness_tau': 0.0,  'sampling_mode': 'preserve', 'screening': 'corr'},
    {'faithfulness_tau': 0.05, 'sampling_mode': 'preserve', 'screening': 'corr'},
    {'faithfulness_tau': 0.05, 'sampling_mode': 'preserve', 'screening': 'pcorr'},
    {'faithfulness_tau': 0.05, 'sampling_mode': 'pool',     'screening': 'corr'},
]

records = []
for cfg in configs:
    t0 = time.time()
    _, G, obj = dag_coordinate_descent_l0_weakfaith(
        S, T=20000, seed=0, lambda_l0=0.2,
        early_stop=True, tol=1e-4, patience=10,
        **cfg,
    )
    elapsed = time.time() - t0
    records.append({**cfg, 'runtime_s': round(elapsed, 2),
                    'shd_directed': shd_score(B_true, G),
                    'final_f': round(obj, 3)})

pd.DataFrame(records)